In [3]:
import os
import json
import pandas as pd
from datasets import Dataset, Audio, Features, Value
from tqdm import tqdm
import math

In [4]:
source_dir = "Hindi_Simple"
output_dir = "Processed_Hindi_Parquet_Dataset"

def fix(v):
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, (str, int, float, bool)):
        return v
    return str(v)

def load_dataset_local(path):
    data = []
    files = [f for f in os.listdir(path) if f.endswith(".json")]

    for fjson in tqdm(files, desc="Loading records"):
        jpath = os.path.join(path, fjson)
        wpath = jpath.replace(".json", ".wav")

        if not os.path.exists(wpath):
            continue

        try:
            with open(jpath, "r") as f:
                meta = json.load(f)
        except:
            continue

        record = {
            "audio": {"path": wpath},
            "language": fix(meta.get("language")),
            "text": fix(meta.get("transcript")),
            "gender": fix(meta.get("gender")),
            "district": fix(meta.get("district")),
            "state": fix(meta.get("state"))
        }

        record = {k: v for k, v in record.items() if v is not None}
        data.append(record)

    df = pd.DataFrame(data)

    features = Features({
        "audio": Audio(sampling_rate=16000),
        "language": Value("string"),
        "text": Value("string"),
        "gender": Value("string"),
        "district": Value("string"),
        "state": Value("string")
    })

    return Dataset.from_pandas(df, features=features, preserve_index=False)

ds = load_dataset_local(source_dir)
ds.save_to_disk(output_dir)


Saving the dataset (4/4 shards): 100%|██████████| 5567/5567 [00:01<00:00, 2928.75 examples/s]


In [6]:
source_dir = "Hindi_Simple"

total_seconds = 0.0
count = 0

for fname in tqdm(os.listdir(source_dir), desc="Scanning audio"):
    if fname.endswith(".wav"):
        path = os.path.join(source_dir, fname)
        try:
            data, sr = sf.read(path)
            total_seconds += len(data) / sr
            count += 1
        except:
            pass

total_hours = total_seconds / 3600

print("Total WAV files:", count)
print("Total hours:", total_hours)
print("Average seconds per file:", total_seconds / count)


Scanning audio: 100%|██████████| 11134/11134 [00:04<00:00, 2738.58it/s]

Total WAV files: 5567
Total hours: 13.355839947916719
Average seconds per file: 8.636792493712985
